# 02 — Clean

Clean `igdb_games_raw` **inside DuckDB** (workflow rule — no pandas transform chains) into `games_clean`:

- round `aggregated_rating` (critic) and `rating` (user) to 1 dp, keep as DOUBLE;
- cast the counts to INTEGER;
- derive `decade` from `release_year`;
- trim text columns; drop rows with no critic aggregate (defensive — the pull   already filtered on `aggregated_rating != null & count >= 3`);
- dedupe on game `id` (keep the one with the most critic scores behind it);
- `quality_report()` before saving; interim saved as **Parquet**.

Both ratings are on the same 0–100 scale but are **separate measures** (critic vs user) — kept side by side, never blended. The critic aggregate is the metric the histogram plots; both feed the critic-vs-user scatter.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql, quality_report, save_interim

cfg = load_config('config.yaml')
con = get_connection(cfg)
print('rows in raw:', con.execute('SELECT COUNT(*) FROM igdb_games_raw').fetchone()[0])

## Build `games_clean` in DuckDB

One SQL pass: round the two ratings, cast counts + year, derive `decade`, TRIM text, keep only rows with a critic aggregate in 0–100, dedupe on `id` (keep the row with the most critic outlets). `decade` supports the era-binning caveat from `SOURCES.md` (aggregates drift over time).

In [ ]:
con.execute('DROP TABLE IF EXISTS games_clean')
con.execute('''
CREATE TABLE games_clean AS
WITH casted AS (
    SELECT
        CAST(id AS BIGINT)                                   AS id,
        TRIM(slug)                                           AS slug,
        TRIM(name)                                           AS name,
        ROUND(TRY_CAST(aggregated_rating AS DOUBLE), 1)      AS critic_rating,
        TRY_CAST(aggregated_rating_count AS INTEGER)         AS critic_count,
        ROUND(TRY_CAST(rating AS DOUBLE), 1)                 AS user_rating,
        TRY_CAST(rating_count AS INTEGER)                    AS user_count,
        TRY_CAST(release_year AS INTEGER)                    AS release_year,
        TRIM(genres)                                         AS genres,
        TRIM(platforms)                                      AS platforms
    FROM igdb_games_raw
),
filtered AS (
    SELECT *,
        CASE WHEN release_year IS NOT NULL
             THEN (release_year // 10) * 10 END              AS decade
    FROM casted
    WHERE critic_rating IS NOT NULL AND critic_rating BETWEEN 0 AND 100
),
deduped AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY critic_count DESC NULLS LAST) AS rn
    FROM filtered
)
SELECT * EXCLUDE (rn) FROM deduped WHERE rn = 1
''')

run_sql('''SELECT COUNT(*) AS n,
  ROUND(MIN(critic_rating),1) mn, ROUND(MAX(critic_rating),1) mx,
  ROUND(AVG(critic_rating),2) mean, ROUND(MEDIAN(critic_rating),1) med,
  SUM(CASE WHEN user_rating IS NOT NULL THEN 1 ELSE 0 END) have_user
  FROM games_clean''', con)

## Quality report
Null counts, dupes, and basic stats before we save the interim file.

In [ ]:
df_clean = con.execute('SELECT * FROM games_clean').df()
quality_report(df_clean, 'games_clean', con, required_columns=['id', 'name', 'critic_rating'])
df_clean.head()

## Save interim (Parquet)

In [ ]:
save_interim(df_clean, cfg, 'games_clean')
print('saved interim games_clean:', df_clean.shape)

---
**Next:** `03-prepare.ipynb` — build the export dataset (CSV/Excel/Parquet) + codebook, and the `chart_*` tables the histogram / line / scatter read.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')